### here we goo 
with a paired difference in CpG parameter between dmel and dsim 

In [1]:
import numpy 
import polars as pl
import plotly.express as px

from plotly.subplots import make_subplots
import plotly.graph_objects as go

from pathlib import Path 

from cogent3.app.io import open_data_store
from cogent3.app import get_app

data_dir = Path.cwd().parent / "data"
results_dir = Path.cwd().parent / "results"
fig_dir = Path.cwd().parent / "figures"

In [2]:
model_results = open_data_store(results_dir / "model-results.sqlitedb", mode="r")
load_json_app = get_app("load_json")

In [ ]:
rows = []

for member in model_results:
    result = load_json_app(member)
    lf = result.alt.lf

    row = {
        "member": member,
        "dmel_cpg": lf.get_param_value(par_name="(CG>TG | CG>CA)", edge="dmel"),
        "dsim_cpg": lf.get_param_value(par_name="(CG>TG | CG>CA)", edge="dsim"),
        "dmel_len": lf.get_param_value(par_name="length", edge="dmel"),
        "dsim_len": lf.get_param_value(par_name="length", edge="dsim"),
        "pvalue": result.pvalue,
    }

    rows.append(row)
df = pl.DataFrame(rows)

In [ ]:
# paired difference (dsim - dmel)
df = df.with_columns(
    (pl.col("dsim_cpg") - pl.col("dmel_cpg")).alias("cpg_diff")
)

fig = px.histogram(
    df.to_dict(),
    x="cpg_diff",
)

fig.update_layout(
    xaxis_title="dsim - dmel CpG transition parameter difference",
    yaxis_title="Count",
)

fig.show()
fig.write_image(fig_dir / "paired-diff-hist.jpg")


In [ ]:
df_sig_only = df.filter(pl.col("pvalue") <= 0.05)

fig = px.histogram(
    df_sig_only.to_dict(),
    x="cpg_diff",
)

fig.update_layout(
    xaxis_title="dsim - dmel CpG transition parameter difference for sig tests results only",
    yaxis_title="Count",
)

fig.show()
fig.write_image(fig_dir / "paired-diff-hist-sig-only.jpg")

how about plotting the actual parameter estimates for the two species?

In [7]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)

fig.add_trace(
    go.Histogram(x=df["dmel_cpg"], name = "dmel"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df["dsim_cpg"], name="dsim/dyak"),
    row=2, col=1
)

fig.update_layout(height=800, width=600, title_text="cpg deamination parameter estimate")
fig.show()

fig.write_image(fig_dir / "cpg_params_subplots.jpg", format="jpg")

In [8]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.05)

fig.add_trace(
    go.Histogram(x=df_sig_only["dmel_cpg"], name = "dmel"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df_sig_only["dsim_cpg"], name="dsim/dyak"),
    row=2, col=1
)

fig.update_layout(height=800, width=600, title_text="cpg deamination parameter estimates for significant lrt only")
fig.show()

fig.write_image(fig_dir / "cpg_params_significant_subplots.jpg", format="jpg")

In [9]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.05)

fig.add_trace(
    go.Histogram(x=df["dmel_len"], name = "dmel", nbinsx=60),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df["dsim_len"], name="dsim", nbinsx=60),
    row=2, col=1
)

fig.update_layout(height=800, width=600, title_text="length parameter estimate")
fig.show()

fig.write_image(fig_dir / "branch_len.jpg", format="jpg")